<a href="https://colab.research.google.com/github/raarchive-tech/e-learning-analytics-etl-bi/blob/main/etl_scripts/elearning_etl_process.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Menetapkan direktori utama upload default Google Colab
base_path = '/content/'

print("--- MENGECEK ISI FOLDER GOOGLE COLAB ---")
if os.path.exists(base_path):
    files = os.listdir(base_path)
    print(f"Direktori '{base_path}' DITEMUKAN.")
    print("Berikut daftar file yang ada di dalam panel kiri kamu saat ini:\n")

    # Menampilkan semua file yang terdeteksi
    for file in files:
        # Mengabaikan folder bawaan .config agar fokus ke dataset kamu
        if file != '.config' and file != 'sample_data':
            print(f"✅ Ada file: {file}")
else:
    print(f"❌ Direktori '{base_path}' tidak ditemukan atau sedang tidak aktif.")

--- MENGECEK ISI FOLDER GOOGLE COLAB ---
Direktori '/content/' DITEMUKAN.
Berikut daftar file yang ada di dalam panel kiri kamu saat ini:

✅ Ada file: pendaftaran_course.csv
✅ Ada file: detail_aktivitas_belajar.xlsx
✅ Ada file: master_course.xlsx
✅ Ada file: data_instruktur.xlsx
✅ Ada file: data_mahasiswa.csv


In [ ]:
import os
import pandas as pd

# 1. Tentukan folder utama tempat kamu upload file di Google Colab
base_dir = '/content/'
files = os.listdir(base_dir)

print("--- TAHAP 1: MENDETEKSI & MEMBACA MULTI-SOURCE FILE ---")

# Trik otomatis mencari nama file berdasarkan kata kunci (biar gak error spasi)
file_mahasiswa = [f for f in files if 'mahasiswa' in f and f.endswith('.csv')][0]
file_pendaftaran = [f for f in files if 'pendaftaran' in f and f.endswith('.csv')][0]
file_instruktur = [f for f in files if 'instruktur' in f and f.endswith('.xlsx')][0]
file_aktivitas = [f for f in files if 'aktivitas' in f and f.endswith('.xlsx')][0]
file_course = [f for f in files if 'master_course' in f and f.endswith('.xlsx')][0]

# Membaca data sesuai ekstensi jeroan aslinya
# File yang murni teks CSV dari awal:
df_mahasiswa = pd.read_csv(os.path.join(base_dir, file_mahasiswa))
df_pendaftaran = pd.read_csv(os.path.join(base_dir, file_pendaftaran))

# File yang jeroannya masih biner Excel (meski namanya ada .csv, kita baca pakai read_excel):
df_instruktur = pd.read_excel(os.path.join(base_dir, file_instruktur), engine='openpyxl')
df_aktivitas = pd.read_excel(os.path.join(base_dir, file_aktivitas), engine='openpyxl')
df_course = pd.read_excel(os.path.join(base_dir, file_course), engine='openpyxl')

print("✅ Sukses membaca ke-5 file!")

print("\n--- TAHAP 2: MENYAMAKAN & MENYIMPAN FORMAT MENJADI CSV MURNI ---")

# Menyimpan ulang file Excel tadi ke format CSV murni di folder Colab
df_instruktur.to_csv('/content/data_instruktur_clean.csv', index=False)
df_aktivitas.to_csv('/content/detail_aktivitas_clean.csv', index=False)
df_course.to_csv('/content/master_course_clean.csv', index=False)
df_mahasiswa.to_csv('/content/data_mahasiswa_clean.csv', index=False)
df_pendaftaran.to_csv('/content/pendaftaran_course_clean.csv', index=False)

print("✅ Sukses! Sekarang semua file sudah seragam menjadi CSV murni.")
print("Silakan cek panel kiri Colab kamu, akan muncul file-file baru berakhiran '_clean.csv'")

--- TAHAP 1: MENDETEKSI & MEMBACA MULTI-SOURCE FILE ---
✅ Sukses membaca ke-5 file!

--- TAHAP 2: MENYAMAKAN & MENYIMPAN FORMAT MENJADI CSV MURNI ---
✅ Sukses! Sekarang semua file sudah seragam menjadi CSV murni.
Silakan cek panel kiri Colab kamu, akan muncul file-file baru berakhiran '_clean.csv'


In [ ]:
print('--- LANGKAH 2: PEMBERSIHAN DATA MAHASISWA ---')

df_mahasiswa = pd.read_csv('/content/data_mahasiswa_clean.csv')

# 1. Menghilangkan spasi tak terlihat di awal/akhir teks kolom
df_mahasiswa['kota'] = df_mahasiswa['kota'].str.strip()
df_mahasiswa['gender'] = df_mahasiswa['gender'].str.strip()

# 2. Memperbaiki typo & singkatan kota menggunakan mapping (pemetaan)
df_mahasiswa['kota'] = df_mahasiswa['kota'].replace({
    'bdg': 'Bandung',
    'Bandung': 'Bandung',
    'jakarta': 'Jakarta',
    'jkt': 'Jakarta',
    'JKT': 'Jakarta',
    'Jakarta': 'Jakarta',
    'bekasi': 'Bekasi',
    'Bekasi': 'Bekasi',
    'Depok': 'Depok'
})

# 3. Menyeragamkan kategori gender
df_mahasiswa['gender'] = df_mahasiswa['gender'].replace({
    'L': 'Male',
    'Laki-laki': 'Male',
    'P': 'Female',
    'female': 'Female'
})

# 4. Menghapus duplikat 'student_id'
df_mahasiswa = df_mahasiswa.drop_duplicates(subset=['student_id'], keep = 'first')

print('Data mahasiswa berhasil dibersihkan')
print(f'jumlah baris setelah dibersihkan: {len(df_mahasiswa)} baris')
df_mahasiswa.head(20)

--- LANGKAH 2: PEMBERSIHAN DATA MAHASISWA ---
Data mahasiswa berhasil dibersihkan
jumlah baris setelah dibersihkan: 500 baris


,student_id,nama_mahasiswa,kota,gender
0,S001,Abyasa Rajata,Bandung,Female
1,S002,"Dr. Diah Prabowo, S.IP",Jakarta,Male
2,S003,Cut Yessi Tampubolon,Bekasi,Male
3,S004,Karimah Simanjuntak,Bekasi,Female
4,S005,Ganjaran Najmudin,Bandung,Male
5,S006,Hartana Pratama,Jakarta,Male
6,S007,Waluyo Padmasari,Bandung,Male
7,S008,Martana Samosir,Jakarta,Male
8,S009,Vino Lazuardi,Depok,Female
9,S010,"Tgk. Ega Napitupulu, S.Kom",Bandung,Female


In [ ]:
print('--- LANGKAH 3: PEMBERSIHAN DATA COURSE & AKTIVITAS ---')

df_course = pd.read_csv('/content/master_course_clean.csv')
df_aktivitas = pd.read_csv('/content/detail_aktivitas_clean.csv')

# COURSE
# 1. Menghapus spasi gaib dan mengubah tulisan kategori menjadi kapital di awal kata (Title Case)
df_course['kategori_course'] = df_course['kategori_course'].str.strip().str.title()

# Mengisi missing value pada 'durasi_jam' dengan nilai median
median_durasi = df_course['durasi_jam'].median()
df_course['durasi_jam'] = df_course['durasi_jam'].fillna(median_durasi)

# AKTIVITAS
df_aktivitas['jenis_aktivitas'] = df_aktivitas['jenis_aktivitas'].str.strip().str.title()

# Mengisi missing value kolom 'nilai' dengan angka 0
df_aktivitas['nilai'] = df_aktivitas['nilai'].fillna(0)

print("✅ Pembersihan Master Course & Detail Aktivitas Selesai!")
print("\n--- Cek Unik Kategori Course (Harus Rapi): ---")
print(df_course['kategori_course'].unique())

print("\n--- Cek Unik Jenis Aktivitas (Harus Rapi): ---")
print(df_aktivitas['jenis_aktivitas'].unique())

print("\n--- Cek Sisa Data Kosong di Kolom Nilai (Harus 0): ---")
print(f"Jumlah data kosong di kolom nilai: {df_aktivitas['nilai'].isna().sum()}")

--- LANGKAH 3: PEMBERSIHAN DATA COURSE & AKTIVITAS ---
✅ Pembersihan Master Course & Detail Aktivitas Selesai!

--- Cek Unik Kategori Course (Harus Rapi): ---
['Data Science' 'Programming' 'Ui/Ux' 'Cyber Security' 'Ai']

--- Cek Unik Jenis Aktivitas (Harus Rapi): ---
['Video Watch' 'Quiz' 'Discussion' 'Assignment' 'Exam']

--- Cek Sisa Data Kosong di Kolom Nilai (Harus 0): ---
Jumlah data kosong di kolom nilai: 0


In [ ]:
print('--- LANGKAH 4: PEMBERSIHAN DATA PENDAFTARAN (TANGGAL & DUPLIKAT) ---')

# 1. Load data
df_pendaftaran = pd.read_csv('/content/pendaftaran_course_clean.csv')

# 2. Bersihkan spasi
df_pendaftaran.columns = df_pendaftaran.columns.str.strip()
df_pendaftaran['tanggal_daftar'] = df_pendaftaran['tanggal_daftar'].str.strip()

# 3. Format tanggal
df_pendaftaran['tanggal_daftar'] = pd.to_datetime(df_pendaftaran['tanggal_daftar'], format='mixed')

# 4. Menghapus duplikat transaksi berdasarkan 'enrollment_id' (Menyimpan baris pertama yang muncul)
jumlah_duplikat = df_pendaftaran.duplicated(subset=['enrollment_id']).sum()
df_pendaftaran = df_pendaftaran.drop_duplicates(subset=['enrollment_id'], keep='first')

print(f"✅ Pembersihan data pendaftaran berhasil!")
print(f"Ditemukan dan dihapus sebanyak {jumlah_duplikat} data pendaftaran duplikat.")
print(f"Jumlah baris pendaftaran sekarang: {len(df_pendaftaran)} baris.")
print("\nSampel data tanggal yang SEKARANG SUDAH SERAGAM (YYYY-MM-DD):")
df_pendaftaran.head(10)

--- LANGKAH 4: PEMBERSIHAN DATA PENDAFTARAN (TANGGAL & DUPLIKAT) ---
✅ Pembersihan data pendaftaran berhasil!
Ditemukan dan dihapus sebanyak 76 data pendaftaran duplikat.
Jumlah baris pendaftaran sekarang: 3000 baris.

Sampel data tanggal yang SEKARANG SUDAH SERAGAM (YYYY-MM-DD):


,enrollment_id,tanggal_daftar,student_id,course_id,instructor_id
0,E00001,2025-02-03,S128,C043,I030
1,E00002,2025-02-28,S420,C017,I016
2,E00003,2025-07-05,S150,C022,I025
3,E00004,2025-06-25,S473,C054,I034
4,E00005,2025-04-06,S370,C002,I009
5,E00006,2025-03-29,S117,C059,I001
6,E00007,2025-04-18,S201,C010,I013
7,E00008,2025-05-24,S445,C047,I014
8,E00009,2025-01-25,S355,C047,I013
9,E00010,2025-02-22,S080,C055,I029


In [ ]:
print("--- LANGKAH 5: MERAKIT STAR SCHEMA & EXPORT AKHIR ---")

# 1. Menggabungkan df_aktivitas dengan df_pendaftaran menggunakan Inner Merge
# Ini akan menyatukan data skor/durasi belajar dengan profil pendaftaran secara presisi
df_fact_merge = pd.merge(df_aktivitas, df_pendaftaran, on='enrollment_id', how='inner')

# 2. Menyusun struktur kolom final untuk Fact Table Elearning
fact_elearning = df_fact_merge[[
    'activity_id', 'enrollment_id', 'student_id', 'course_id',
    'instructor_id', 'jenis_aktivitas', 'nilai', 'durasi_menit', 'tanggal_daftar'
]]

print("✅ Fact Table (fact_aktivitas_elearning) berhasil dirakit!")
print(f"Total baris transaksi dalam Tabel Fakta: {len(fact_elearning)} baris.")

print("\n--- PROSES EXPORT DATA BERSIH KE CSV ---")

# 3. Export semua komponen Star Schema ke file CSV baru untuk siap ditarik ke Power BI / Looker Studio
df_mahasiswa.to_csv('/content/dim_mahasiswa.csv', index=False)
df_course.to_csv('/content/dim_course.csv', index=False)
df_instruktur.to_csv('/content/dim_instruktur.csv', index=False)
fact_elearning.to_csv('/content/fact_aktivitas_elearning.csv', index=False)

print("\n🚀 ETL PIPELINE SUKSES TOTAL!")
print("File berikut sudah siap di panel kiri Colab kamu untuk bahan Dashboard BI:")
print("1. dim_mahasiswa.csv (Dimensi Profil Mahasiswa)")
print("2. dim_course.csv (Dimensi Ragam Course)")
print("3. dim_instruktur.csv (Dimensi Data Pengajar)")
print("4. fact_aktivitas_elearning.csv (Tabel Fakta Aktivitas Belajar)")

--- LANGKAH 5: MERAKIT STAR SCHEMA & EXPORT AKHIR ---
✅ Fact Table (fact_aktivitas_elearning) berhasil dirakit!
Total baris transaksi dalam Tabel Fakta: 9000 baris.

--- PROSES EXPORT DATA BERSIH KE CSV ---

🚀 ETL PIPELINE SUKSES TOTAL!
File berikut sudah siap di panel kiri Colab kamu untuk bahan Dashboard BI:
1. dim_mahasiswa.csv (Dimensi Profil Mahasiswa)
2. dim_course.csv (Dimensi Ragam Course)
3. dim_instruktur.csv (Dimensi Data Pengajar)
4. fact_aktivitas_elearning.csv (Tabel Fakta Aktivitas Belajar)
